# ResNet-152: export probabilities for the ConvNeXt ensemble

No training and no changes to `ResNet.ipynb`. This reproduces its deterministic
224-pixel validation preprocessing and sigmoid predictions from the saved best
Lightning checkpoint. No TTA and no `keep_one` postprocessing.

Attach the competition, the colleague's **best `.ckpt`**, and the **actual `folds.csv`
used to train that checkpoint**. Set `CHECKPOINT_PATH` and `FOLDS_PATH` below.
The repository's ResNet trains with held-out fold 0; change `RESNET_VAL_FOLD`
only if the colleague actually trained a different split. A checkpoint alone does
not prove which rows were held out; the split declaration must be accurate.

The exported bundle contains validation probabilities, test probabilities, and
their provenance. Share the entire `resnet_for_ensemble` folder with the teammate.

In [ ]:
# Only if imports fail; run once, before the imports below.
# %pip install -q timm albumentations


## Imports and paths
Use your colleague's trained weights; no ImageNet download or Lightning installation is needed.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display, FileLink

CHECKPOINT_PATH = None  # e.g. Path('/kaggle/input/resnet-backup/resnet152-epoch.ckpt')
FOLDS_PATH = None       # e.g. Path('/kaggle/input/resnet-backup/folds.csv')
RESNET_VAL_FOLD = 0     # actual held-out fold used for this checkpoint
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working/resnet_for_ensemble')
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
MODEL_NAME = 'resnet152.a1_in1k'


## Validate the original split
The full split is checked against competition labels; predictions will be made only
for the declared held-out fold and for test images. Training images are not evaluated.

In [ ]:
def read_unique(path):
    frame = pd.read_csv(path, dtype={'image_id': str})
    assert len(frame) and frame.image_id.notna().all() and frame.image_id.is_unique, str(path)
    return frame

def align(frame, ids):
    ids = list(ids)
    assert len(ids) == len(set(ids)) and set(frame.image_id) == set(ids), 'Image IDs differ.'
    return frame.set_index('image_id').loc[ids].reset_index()

def probabilities(frame, columns):
    values = frame[columns].to_numpy(dtype=np.float64)
    assert values.shape == (len(frame), len(LABEL_COLS)) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all(), 'Expected probabilities in [0, 1].'
    return values

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def folds_digest(frame):
    canonical = frame[['image_id', *LABEL_COLS, 'cluster', 'fold']].sort_values('image_id').copy()
    canonical[['cluster', 'fold', *LABEL_COLS]] = canonical[['cluster', 'fold', *LABEL_COLS]].astype('int64')
    return hashlib.sha256(canonical.to_csv(index=False, lineterminator=chr(10)).encode()).hexdigest()

def find_directories(required_files):
    found = []
    for root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        if not root.is_dir():
            continue
        for directory, subdirs, files in os.walk(root):
            subdirs[:] = [name for name in subdirs if name not in {'train', 'test', 'images', 'wandb', '.virtual_documents'}]
            if set(required_files).issubset(files):
                found.append(Path(directory))
    return found

def resolve_directory(preferred, required_files, predicate=lambda directory: True):
    preferred = Path(preferred) if preferred is not None else None
    if preferred is not None and all((preferred / name).is_file() for name in required_files):
        assert predicate(preferred), f'Incompatible files in {preferred}'
        return preferred
    candidates = [directory for directory in find_directories(required_files) if predicate(directory)]
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Set the directory explicitly; attach the saved Output or upload the backup. '
            f'Required: {required_files}. Candidates: {candidates}'
        )
    return candidates[0]

DATA_DIR = resolve_directory(DATA_DIR, ['train.csv', 'test.csv', 'sample_submission.csv'])
if CHECKPOINT_PATH is None or FOLDS_PATH is None:
    raise ValueError('Set CHECKPOINT_PATH to the best ResNet .ckpt and FOLDS_PATH to its original folds.csv.')
CHECKPOINT_PATH, FOLDS_PATH = Path(CHECKPOINT_PATH), Path(FOLDS_PATH)
assert CHECKPOINT_PATH.is_file() and FOLDS_PATH.is_file()
assert RESNET_VAL_FOLD in range(5)
train = read_unique(DATA_DIR / 'train.csv')
folds = align(read_unique(FOLDS_PATH), train.image_id)
assert {'cluster', 'fold', *LABEL_COLS}.issubset(folds.columns)
assert train[LABEL_COLS].isin([0, 1]).all().all()
assert np.array_equal(train[LABEL_COLS].to_numpy(), folds[LABEL_COLS].to_numpy()), 'Stale folds labels.'
assert set(folds.fold.unique()) == set(range(5))
assert folds.groupby('cluster').fold.nunique().eq(1).all(), 'A duplicate group crosses folds.'
val_frame = folds.loc[folds.fold == RESNET_VAL_FOLD].reset_index(drop=True)
training_frame = folds.loc[folds.fold != RESNET_VAL_FOLD]
assert len(val_frame) and len(training_frame)
assert set(val_frame.cluster).isdisjoint(training_frame.cluster)
sample = read_unique(DATA_DIR / 'sample_submission.csv')
test = read_unique(DATA_DIR / 'test.csv')
assert set(sample.columns) == {'image_id', *LABEL_COLS}
assert set(sample.image_id) == set(test.image_id)
print('Device:', DEVICE, '| held-out fold:', RESNET_VAL_FOLD, '| validation:', len(val_frame), '| test:', len(sample))


## Restore the model and deterministic preprocessing
`pretrained=False` avoids downloading initialization: the full trained model is
restored strictly from the colleague's Lightning `state_dict` (`model.*` keys).
Load only the checkpoint supplied by your colleague.

In [ ]:
checkpoint_sha256 = file_digest(CHECKPOINT_PATH)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert isinstance(checkpoint, dict) and 'state_dict' in checkpoint, 'Expected the Lightning checkpoint from ResNet.ipynb.'
state = {key[len('model.'):]: value for key, value in checkpoint['state_dict'].items() if key.startswith('model.')}
assert state, 'No model.* weights found in the checkpoint.'
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABEL_COLS))
model.load_state_dict(state, strict=True)
model = model.to(DEVICE).eval()
del checkpoint, state

preprocessing = {'name': 'albumentations_resize_v1', 'image_size': 224,
                 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'interpolation': 1}
transform = A.Compose([
    A.Resize(224, 224, interpolation=1),
    A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
    ToTensorV2(),
])

class PredictionDataset(Dataset):
    def __init__(self, frame, image_dir):
        self.ids = frame.image_id.tolist()
        self.image_dir = Path(image_dir)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(self.image_dir / f'{image_id}.jpg') as image:
            image = np.array(image.convert('RGB'))
        return transform(image=image)['image'], image_id

@torch.inference_mode()
def predict(frame, image_dir, description):
    loader = DataLoader(PredictionDataset(frame, image_dir), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    parts, ordered_ids = [], []
    for images, image_ids in tqdm(loader, desc=description):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        parts.append(logits.float().sigmoid().cpu().numpy())
        ordered_ids.extend(image_ids)
    assert ordered_ids == frame.image_id.tolist()
    result = pd.DataFrame(np.concatenate(parts), columns=LABEL_COLS)
    result.insert(0, 'image_id', ordered_ids)
    probabilities(result, LABEL_COLS)
    return result


## Export validation and test probabilities
Keep all three files together. The manifest pins checkpoint, split and CSV hashes;
the ensemble notebook verifies them before using the probabilities.

In [ ]:
validation_probabilities = predict(val_frame, DATA_DIR / 'train', 'ResNet validation')
test_probabilities = predict(sample, DATA_DIR / 'test', 'ResNet test')
validation = val_frame[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
for label in LABEL_COLS:
    validation[label + '_prob'] = validation_probabilities[label].to_numpy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
validation_path = OUTPUT_DIR / 'validation_predictions.csv'
test_path = OUTPUT_DIR / 'test_probabilities.csv'
validation.to_csv(validation_path, index=False)
test_probabilities.to_csv(test_path, index=False)
manifest = {
    'schema': 'resnet_probability_export_v1', 'model_name': MODEL_NAME, 'label_cols': LABEL_COLS,
    'held_out_fold': RESNET_VAL_FOLD, 'folds_fingerprint': folds_digest(folds),
    'checkpoint_sha256': checkpoint_sha256, 'checkpoint_filename': CHECKPOINT_PATH.name,
    'split_source': 'original training folds.csv; held-out fold declared by checkpoint owner',
    'preprocessing': preprocessing, 'tta': False, 'postprocessing': 'none',
    'file_hashes': {validation_path.name: file_digest(validation_path), test_path.name: file_digest(test_path)},
}
(OUTPUT_DIR / 'resnet_export.json').write_text(json.dumps(manifest, indent=2))
from sklearn.metrics import f1_score
print('ResNet held-out macro F1 at 0.5:', f1_score(
    validation[LABEL_COLS], probabilities(validation, [label + '_prob' for label in LABEL_COLS]) >= 0.5,
    average='macro', zero_division=0))
print('Share the entire folder:', OUTPUT_DIR)
display(validation.head())


## Optional: download the export bundle
This small archive contains probabilities and metadata, not the large checkpoint.
Refresh Kaggle Output and download it; upload it as a private Dataset for the ensemble.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive = OUTPUT_DIR.parent / 'resnet_for_ensemble.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as bundle:
    for name in ['validation_predictions.csv', 'test_probabilities.csv', 'resnet_export.json']:
        bundle.write(OUTPUT_DIR / name, arcname=f'{OUTPUT_DIR.name}/{name}')
print('Saved:', archive)
display(FileLink(str(archive)))
